In [ ]:
import pandas as pd
from utils.ml_utils_v2 import EnsembleProjections
from utils.eda_utils import FeatureEngineering, EDAUtils, Filters
import os
import joblib

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
ep = EnsembleProjections()

In [111]:
# Set up paths
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")
RESULTS_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "results")
OUTPUT_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "output")
ENSEMBLE_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "ensemble")
MODELS_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "models")
TRAINING_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "training")
POST_PROCESSING_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "2030_emissions")

In [ ]:
# Ensemble params
n_scenarios = 100

# Load data
training_df_log_transformed = pd.read_csv(os.path.join(TRAINING_DIR_PATH, "training_df_log_transformed.csv"))
training_df_lags = pd.read_csv(os.path.join(TRAINING_DIR_PATH, "training_df_lags.csv"))
ensemble_arima_df = pd.read_parquet(os.path.join(ENSEMBLE_DIR_PATH, f"ensemble_arima_{n_scenarios}.parquet"))

In [ ]:
# Print shapes
print(f"Training df log transformed shape: {training_df_log_transformed.shape}")
print(f"Training df lags shape: {training_df_lags.shape}")
print(f"Ensemble ARIMA df shape: {ensemble_arima_df.shape}") 

In [ ]:
# Get the models from the models directory with keys without .pkl extension
models = {}
for model_name in os.listdir(MODELS_DIR_PATH):
    if model_name.endswith(".pkl"):
        model_path = os.path.join(MODELS_DIR_PATH, model_name)
        key_name = model_name[:-4]  # remove .pkl extension
        models[key_name] = joblib.load(model_path)

models.keys()

In [ ]:
# Get a df with only iso_alpha_3, income_group and region cols
iso_alpha_3_cols = ["iso_alpha_3", "income_group", "region"]
income_group_region_mapping_df = training_df_log_transformed[iso_alpha_3_cols].drop_duplicates().reset_index(drop=True)
income_group_region_mapping_df.head()

In [ ]:
emission_init_cond_df = training_df_lags[training_df_lags["year"] == 2022].copy()
emission_init_cond_df = emission_init_cond_df.reset_index(drop=True)
emission_init_cond_df = emission_init_cond_df[["iso_alpha_3", "year", "total_emissions"]]
emission_init_cond_df

In [ ]:
training_df_log_transformed.columns

In [ ]:
features_init_cond_df = training_df_log_transformed[training_df_log_transformed["year"] == 2022].copy()
features_init_cond_df = features_init_cond_df.reset_index(drop=True)
features_init_cond_df = features_init_cond_df.drop(columns=["income_group", "region"])
features_init_cond_df

In [ ]:
ensemble_arima_df.head()

In [ ]:
# Merge the income group and region mapping df with the ensemble df
ensemble_arima_df = pd.merge(ensemble_arima_df, income_group_region_mapping_df,
                             on="iso_alpha_3", how="left")

ensemble_arima_df.head()

## Let's check the feature variables projections

In [ ]:
ep.plot_ensemble_time_series(
    df=ensemble_arima_df, 
    iso_alpha_3="USA",
    column="log_pop_total",
    hist_df=training_df_log_transformed)

In [ ]:
ep.plot_ensemble_time_series(
    df=ensemble_arima_df, 
    iso_alpha_3="USA",
    column="log_industry_pct_of_gdp",
    hist_df=training_df_log_transformed)

### We need to rescale the features so they match with the intial year (2022)

In [ ]:
ensemble_arima_df.head()

In [ ]:
calibrated_ensemble_arima_df = ep.calibrate_to_initial_conditions(
    simulated_df=ensemble_arima_df,
    initial_conditions_df=features_init_cond_df,  # your 2022 table
    base_year=2022,
    # columns=None -> auto-intersection
    update_logs=True,                 # keeps log_* in sync if present
    log_prefix="log_"
)


In [ ]:
ep.plot_ensemble_time_series(
    df=calibrated_ensemble_arima_df, 
    iso_alpha_3="USA",
    column="log_pop_total",
    hist_df=training_df_log_transformed)

In [ ]:
ep.plot_ensemble_time_series(
    df=calibrated_ensemble_arima_df, 
    iso_alpha_3="USA",
    column="log_industry_pct_of_gdp",
    hist_df=training_df_log_transformed)

## Let's do a prediction and see how it looks without rescaling

In [ ]:
# Get feature cols
feature_cols_no_isos = [c for c in  training_df_log_transformed.columns if c not in ["iso_alpha_3", "log_total_emissions"]]
feature_cols_no_isos

In [ ]:
feature_cols_with_isos = [c for c in training_df_log_transformed.columns if c not in ["log_total_emissions"]]
feature_cols_with_isos

In [ ]:
enet_no_isos_df = ep.predict_ensemble_emissions(calibrated_ensemble_arima_df, models["reg_no_isos_enet_pipeline"], feature_cols=feature_cols_no_isos)
enet_with_isos_df = ep.predict_ensemble_emissions(calibrated_ensemble_arima_df, models["reg_with_isos_enet_pipeline"], feature_cols=feature_cols_with_isos)


In [ ]:
ep.plot_ensemble_time_series(
    df=enet_no_isos_df, 
    iso_alpha_3="USA",
    column="total_emissions",
    hist_df=training_df_lags)

In [ ]:
ep.plot_ensemble_time_series(
    df=enet_with_isos_df, 
    iso_alpha_3="MEX",
    column="total_emissions",
    hist_df=training_df_lags)

In [ ]:
ep.plot_ensemble_time_series(
    df=enet_no_isos_df, 
    iso_alpha_3="MEX",
    column="total_emissions",
    hist_df=training_df_lags)

## Let's apply an HP filter to smooth the trayectories

In [ ]:
fe = FeatureEngineering()
edau = EDAUtils()
filter_utils = Filters()

In [ ]:
historical_emissions_df = training_df_lags[["iso_alpha_3", "year", "total_emissions"]].copy()
historical_emissions_df.head()

In [ ]:
historical_emissions_df.year.max()

In [ ]:
hist_em_hp_trend_df = fe.hp_filter_panel(historical_emissions_df, which="trend", lambda_=20, keep="filtered_only")
hist_em_hp_trend_df.head()

In [ ]:
edau.plot_iso_numeric_subplots(historical_emissions_df, "MEX")
edau.plot_iso_numeric_subplots(hist_em_hp_trend_df, "MEX")

In [ ]:
enet_no_isos_hp_trend_df = filter_utils.hp_filter_panel_fast(enet_no_isos_df, 
                                                            which="trend", 
                                                            lambda_=20,
                                                            keep="filtered_only", 
                                                            id_col="future_id",
                                                            n_jobs=-1)

In [ ]:
ep.plot_ensemble_time_series(
    df=enet_no_isos_hp_trend_df,
    iso_alpha_3="USA",
    column="total_emissions_hp_trend",
    hist_df=hist_em_hp_trend_df)

In [ ]:
emission_init_cond_hp_trend_df = hist_em_hp_trend_df[hist_em_hp_trend_df["year"] == 2022].copy()
emission_init_cond_hp_trend_df = emission_init_cond_hp_trend_df.reset_index(drop=True)
emission_init_cond_hp_trend_df = emission_init_cond_hp_trend_df[["iso_alpha_3", "year", "total_emissions_hp_trend"]]
emission_init_cond_hp_trend_df

In [ ]:
enet_no_isos_hp_trend_df_calibrated = ep.calibrate_to_initial_conditions(
    simulated_df=enet_no_isos_hp_trend_df,
    initial_conditions_df=emission_init_cond_hp_trend_df,
    base_year=2022,
    adjustment_method="multiplicative"
)

In [ ]:
ep.plot_ensemble_time_series(
    df=enet_no_isos_hp_trend_df_calibrated,
    iso_alpha_3="UGA",
    column="total_emissions_hp_trend",
    hist_df=hist_em_hp_trend_df)

## Dealing with Outliers

In [ ]:
emissions_df = enet_no_isos_hp_trend_df_calibrated[["future_id","iso_alpha_3", "year", "total_emissions_hp_trend"]].copy()

In [ ]:
emissions_df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_histograms_by_country(df: pd.DataFrame, year: int = 2030):
    """
    Plot a histogram of total_emissions_hp_trend for each iso_alpha_3
    for the specified year.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns ['future_id', 'iso_alpha_3', 'year', 'total_emissions_hp_trend'].
    year : int, optional
        The year to filter the data on. Default is 2030.
    """
    # Filter for the given year
    year_df = df[df['year'] == year]
    
    # Get unique country codes
    countries = year_df['iso_alpha_3'].unique()
    
    # Create a histogram for each country
    for country in countries:
        country_data = year_df[year_df['iso_alpha_3'] == country]['total_emissions_hp_trend']
        
        plt.figure(figsize=(8, 5))
        plt.hist(country_data, bins=20, color='skyblue', edgecolor='black')
        plt.title(f'{country} - Total Emissions in {year}')
        plt.xlabel('Total Emissions (hp trend)')
        plt.ylabel('Frequency')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()


In [ ]:
# Assuming your DataFrame is named df
# plot_histograms_by_country(emissions_df, year=2030)

In [ ]:
import pandas as pd
from typing import Tuple, Optional, Union, List

def remove_timeseries_with_year_outliers_iqr(
    df: pd.DataFrame,
    year: int = 2030,
    value_col: str = "total_emissions_hp_trend",
    country_col: str = "iso_alpha_3",
    id_col: str = "future_id",
    iqr_multiplier: float = 1.5,
    min_group_size: int = 5,
    treat_zero_iqr_as_no_outliers: bool = True,
    return_removed_ids: bool = False
) -> Union[pd.DataFrame, Tuple[pd.DataFrame, List[str]]]:
    """
    Identify outliers in `value_col` for a specific `year` per `country_col`
    using the IQR method, then remove the ENTIRE time series for any outlier
    `id_col` (i.e., drop all rows for those future_id values across all years).

    Parameters
    ----------
    df : pd.DataFrame
        Must include columns [id_col, country_col, 'year', value_col].
    year : int
        Target year to detect outliers on (default 2030).
    value_col : str
        Column to analyze for outliers (default 'total_emissions_hp_trend').
    country_col : str
        Country/grouping column (default 'iso_alpha_3').
    id_col : str
        Series identifier (default 'future_id').
    iqr_multiplier : float
        IQR rule multiplier (1.5 is standard; lower is stricter).
    min_group_size : int
        Minimum sample size per country for outlier detection; groups
        smaller than this will not have any outliers removed.
    treat_zero_iqr_as_no_outliers : bool
        If True, when IQR == 0 for a country-year, treat as no outliers.
        If False, bounds collapse to a point and anything != Q1 is outlier.
    return_removed_ids : bool
        If True, also return a list of removed future_id values.

    Returns
    -------
    cleaned_df : pd.DataFrame
        DataFrame with all rows for outlier future_ids removed.
    removed_ids : list[str] (optional)
        Only returned if return_removed_ids=True.
    """
    required = {id_col, country_col, "year", value_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame is missing required columns: {missing}")

    # Only the target year's slice for detecting outliers
    year_df = df[df["year"] == year][[id_col, country_col, value_col]].copy()
    if year_df.empty:
        cleaned = df.copy()
        return (cleaned, []) if return_removed_ids else cleaned

    # Compute Q1, Q3, IQR per country on the target year
    stats = (
        year_df.groupby(country_col)[value_col]
        .agg(q1=lambda s: s.quantile(0.25),
             q3=lambda s: s.quantile(0.75),
             n="count")
        .reset_index()
    )
    stats["iqr"] = stats["q3"] - stats["q1"]

    # Optionally ignore countries with IQR == 0
    if treat_zero_iqr_as_no_outliers:
        stats["use_bounds"] = (stats["iqr"] > 0) & (stats["n"] >= min_group_size)
    else:
        stats["use_bounds"] = (stats["n"] >= min_group_size)

    # Lower/upper bounds
    stats["lower"] = stats["q1"] - iqr_multiplier * stats["iqr"]
    stats["upper"] = stats["q3"] + iqr_multiplier * stats["iqr"]

    # Merge bounds onto the target-year rows
    year_with_bounds = year_df.merge(stats[[country_col, "lower", "upper", "use_bounds"]],
                                     on=country_col, how="left")

    # Determine outliers (only where we decide to use bounds)
    mask_use = year_with_bounds["use_bounds"].fillna(False)
    mask_low = year_with_bounds[value_col] < year_with_bounds["lower"]
    mask_high = year_with_bounds[value_col] > year_with_bounds["upper"]
    outlier_mask = mask_use & (mask_low | mask_high)

    removed_ids = year_with_bounds.loc[outlier_mask, id_col].unique().tolist()

    # Drop whole time series for removed IDs
    cleaned_df = df[~df[id_col].isin(removed_ids)].copy()

    return (cleaned_df, removed_ids) if return_removed_ids else cleaned_df


In [ ]:
# Standard 1.5*IQR rule
clean_df, removed = remove_timeseries_with_year_outliers_iqr(
    emissions_df,
    year=2030,
    iqr_multiplier=1.5,
    return_removed_ids=True
)

print(f"Removed {len(removed)} future_id series")



In [ ]:
clean_df

In [ ]:
ep.plot_ensemble_time_series(
    df=clean_df,
    iso_alpha_3="HRV",
    column="total_emissions_hp_trend",
    hist_df=hist_em_hp_trend_df)

In [112]:
clean_df.to_csv(os.path.join(POST_PROCESSING_DIR_PATH, "post_processed_projected_emissions.csv"), index=False)